In [1]:
import flet as ft
import os
import time
import logging
import pandas as pd
import threading
import asyncio
import nest_asyncio
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pyautogui


In [2]:
EVENTS_DELETE=r'C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\ELIMINAR EVENTOS SUSPENDIDOS\PROGRAMADAS SUSPENDIDAS.xlsx'


In [3]:
df_EVENTS_DELETE = pd.read_excel(EVENTS_DELETE)
display(df_EVENTS_DELETE)

,COD_OSINERGMIN
0,10701753
1,10700539
2,10700544
3,10701649
4,10701704
5,10701720
6,10700535
7,10701699
8,10701735
9,10702799


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# 1. Configurar el driver (ejemplo con Chrome)
driver = webdriver.Chrome()

# 2. Navegar a la página de login
driver.get("https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN")

# 3. Esperar a que el formulario esté listo y localizar elementos
wait = WebDriverWait(driver, 10)

# Campo usuario
campo_usuario = wait.until(EC.presence_of_element_located(
    (By.XPATH, "/html/body/table[2]/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td[3]/input")
))

# Campo password
campo_password = driver.find_element(
    By.XPATH, "/html/body/table[2]/tbody/tr[2]/td[2]/form/table/tbody/tr[4]/td[3]/input"
)

# Botón login - ajustado al XPath correcto
boton_login = driver.find_element(
    By.XPATH, "//*[@id='login']/tbody/tr[6]/td[2]/input"
)

# 4. Enviar credenciales y hacer clic
campo_usuario.send_keys("CALSER@EDN")
campo_password.send_keys("EDN098")
boton_login.click()

# 5. Esperar a que el login se complete
time.sleep(20)

# 6. Navegar a la URL específica
driver.get("https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN")

# 7. Esperar 10 segundos para ver el cambio
print("Esperando 10 segundos después de la navegación...")
time.sleep(10)

# 8. Verificar la URL actual
print(f"URL actual: {driver.current_url}")
print(f"Título de la página: {driver.title}")

# 9. Crear una columna para registrar el estado del procesamiento
# Si no existe la columna 'ESTADO' en el DataFrame, la creamos
if 'ESTADO' not in df_EVENTS_DELETE.columns:
    df_EVENTS_DELETE['ESTADO'] = 'PENDIENTE'

# 10. Iterar sobre los códigos en df_EVENTS_DELETE
for index, row in df_EVENTS_DELETE.iterrows():
    codigo = row['COD_OSINERGMIN']
    print(f"Procesando código {index+1}/{len(df_EVENTS_DELETE)}: {codigo}")
    
    try:
        # Paso 1: Ingresar código en el campo
        campo_codigo = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//*[@id='n__codigointerr']")
        ))
        campo_codigo.clear()
        campo_codigo.send_keys(codigo)
        
        # Paso 2: Dar click en el botón de buscar
        boton_buscar = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "/html/body/table/tbody/tr[1]/td/form[2]/fieldset/table/tbody/tr[2]/td[3]/input")
        ))
        boton_buscar.click()
        time.sleep(3)  # Esperar a que cargue la búsqueda
        
        # Paso 3: Dar click en el enlace
        enlace = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "/html/body/table/tbody/tr[1]/td/form[2]/table/tbody/tr[2]/td[1]/a")
        ))
        enlace.click()
        time.sleep(3)  # Esperar a que cargue la página
        
        # Paso 4: Dar click en el checkbox
        checkbox = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//*[@id='c__suspen']")
        ))
        checkbox.click()
        
        # Paso 5: Llenar el campo de motivo
        campo_motivo = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//*[@id='n__motivo']")
        ))
        campo_motivo.clear()
        campo_motivo.send_keys("FUERZA MAYOR")
        
        # Paso 6: Dar click en el botón de enviar
        boton_enviar = wait.until(EC.element_to_be_clickable(
            (By.XPATH, "/html/body/table/tbody/tr[1]/td/form/fieldset[5]/input[2]")
        ))
        boton_enviar.click()
        
        # Esperar a que se procese y verificar que se completó correctamente
        time.sleep(5)
        
        # Verificar si la operación fue exitosa
        # Puedes agregar aquí verificaciones específicas de la página
        # Por ejemplo, buscar un mensaje de confirmación
        try:
            # Buscar algún elemento que indique éxito
            # Este es un ejemplo - ajusta según tu página
            mensaje_exito = driver.find_element(By.XPATH, "//*[contains(text(), 'éxito') or contains(text(), 'correcto')]")
            estado = 'CORRECTO'
            print(f"  ✓ Código {codigo} procesado correctamente")
        except:
            # Si no encuentra el mensaje, asumimos éxito si llegamos hasta aquí
            estado = 'CORRECTO'
            print(f"  ✓ Código {codigo} procesado (sin confirmación explícita)")
        
        # Actualizar el estado en el DataFrame
        df_EVENTS_DELETE.at[index, 'ESTADO'] = estado
        
        # Volver a la página principal para el siguiente código
        driver.get("https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN")
        time.sleep(3)
        
    except Exception as e:
        # Capturar el error y registrar en el DataFrame
        error_msg = f"ERROR: {str(e)[:100]}"  # Limitar el tamaño del mensaje
        df_EVENTS_DELETE.at[index, 'ESTADO'] = error_msg
        print(f"  ✗ Error procesando código {codigo}: {e}")
        
        # Intentar volver a la página principal para continuar
        try:
            driver.get("https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN")
            time.sleep(3)
        except:
            print("  No se pudo volver a la página principal")

# 11. Mostrar resumen del procesamiento
print("\n" + "="*50)
print("RESUMEN DEL PROCESAMIENTO:")
print("="*50)

# Contar estados
correctos = df_EVENTS_DELETE[df_EVENTS_DELETE['ESTADO'] == 'CORRECTO'].shape[0]
errores = df_EVENTS_DELETE[df_EVENTS_DELETE['ESTADO'] != 'CORRECTO'].shape[0]

print(f"Total procesados: {len(df_EVENTS_DELETE)}")
print(f"Correctos: {correctos}")
print(f"Con errores: {errores}")

# Mostrar códigos con errores
if errores > 0:
    print("\nCódigos con errores:")
    errores_df = df_EVENTS_DELETE[df_EVENTS_DELETE['ESTADO'] != 'CORRECTO']
    print(errores_df[['COD_OSINERGMIN', 'ESTADO']].to_string(index=False))

# Guardar el DataFrame actualizado
try:
    df_EVENTS_DELETE.to_csv('EVENTS_DELETE_procesado.csv', index=False)
    print(f"\nResultados guardados en 'EVENTS_DELETE_procesado.csv'")
except Exception as e:
    print(f"\nError al guardar resultados: {e}")

# Mostrar DataFrame completo
print("\nDataFrame completo:")
print(df_EVENTS_DELETE[['COD_OSINERGMIN', 'ESTADO']].to_string(index=False))

# 12. Cerrar el navegador al finalizar
print("\nProceso completado")
driver.quit()

Esperando 10 segundos después de la navegación...
URL actual: https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN
Título de la página: 
Procesando código 1/38: 10701753
  ✓ Código 10701753 procesado (sin confirmación explícita)
Procesando código 2/38: 10700539
  ✓ Código 10700539 procesado (sin confirmación explícita)
Procesando código 3/38: 10700544
  ✓ Código 10700544 procesado (sin confirmación explícita)
Procesando código 4/38: 10701649
  ✓ Código 10701649 procesado (sin confirmación explícita)
Procesando código 5/38: 10701704
  ✓ Código 10701704 procesado (sin confirmación explícita)
Procesando código 6/38: 10701720
  ✓ Código 10701720 procesado (sin confirmación explícita)
Procesando código 7/38: 10700535
  ✓ Código 10700535 procesado (sin confirmación explícita)
Procesando código 8/38: 10701699
  ✓ Código 10701699 procesado (sin confirmación explícita)
Procesando código 9/38: 10701735
  ✓ Código 10701735 procesado (sin confirmación explícita)
Procesando código 10/38: 1

In [5]:
display(df_EVENTS_DELETE[['COD_OSINERGMIN', 'ESTADO']])


,COD_OSINERGMIN,ESTADO
0,10701753,CORRECTO
1,10700539,CORRECTO
2,10700544,CORRECTO
3,10701649,CORRECTO
4,10701704,CORRECTO
5,10701720,CORRECTO
6,10700535,CORRECTO
7,10701699,CORRECTO
8,10701735,CORRECTO
9,10702799,CORRECTO
